# TipCat Pipeline Manager
Interactive notebook for managing design uploads and running Cloud Run pipeline steps.

In [ ]:
import json
import subprocess
from google.cloud import storage

GCP_PROJECT = "tipcat-automation"
GCS_BUCKET = "tipcat-product-designs"
CLOUD_RUN_JOB = "tipcat-pipeline"
CLOUD_RUN_REGION = "us-central1"

storage_client = storage.Client(project=GCP_PROJECT)
bucket = storage_client.bucket(GCS_BUCKET)
print("Ready:", GCP_PROJECT, GCS_BUCKET)

In [ ]:
def read_generated_metadata():
    blob = bucket.blob("output/generated_metadata.json")
    data = json.loads(blob.download_as_text())
    print(f"Loaded {len(data)} metadata items")
    for item in data[:3]:
        sku = item.get("sku")
        gcs_path = item.get("gcs_path", "")
        title = item.get("analysis", {}).get("metadata", {}).get("title", "")
        print(f"SKU {sku}: {title}")
        if gcs_path:
            print("   ", gcs_path)
    return data

# Example:
# metadata = read_generated_metadata()

In [ ]:
def run_step(step=1, limit=3, verbose=True):
    args = [f"--step={step}"]
    if limit and int(limit) > 0:
        args.append(f"--limit={int(limit)}")
    if verbose:
        args.append("--verbose")

    cmd = [
        "gcloud", "run", "jobs", "execute", CLOUD_RUN_JOB,
        f"--region={CLOUD_RUN_REGION}",
        f"--project={GCP_PROJECT}",
        f"--args={','.join(args)}"
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print("exit:", result.returncode)
    print(result.stdout[:2000])
    if result.stderr:
        print(result.stderr[:2000])

# Example:
# run_step(step=1, limit=3, verbose=True)

In [ ]:
def list_designs(prefix="designs/"):
    blobs = list(bucket.list_blobs(prefix=prefix))
    designs = [b.name for b in blobs if b.name.lower().endswith(".png")]
    print(f"Found {len(designs)} PNG files")
    for name in designs[:20]:
        print(" -", name)
    if len(designs) > 20:
        print(f"... and {len(designs) - 20} more")
    return designs

_ = list_designs()